# 03 - Feature Engineering

This notebook creates rolling team-form features for each match. The key rule is that features for a match may only use games played before. This prevents data leakage, making model evaluation more realistic.

In [2]:
from pathlib import Path
from collections import defaultdict
import json

import pandas as pd
import numpy as np

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [3]:
results = pd.read_csv(PROCESSED_DIR / "clean_matches.csv")
results["date"] = pd.to_datetime(results["date"])

results = results.sort_values("date").reset_index(drop=True)

results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,total_goals,result
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,0,draw
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,6,home_win
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,3,home_win
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,4,draw
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,3,home_win


## Data Leakage Rule

For each match, the model can only use information from matches that happened earlier. If the model accidentally uses future matches, the performance numbers will look better than they really are.

To avoid this, the feature-building loop processes the data in chronological order. For each row, it calculates each team's recent form first, then updates that team's history after the match is processed.

In [4]:
def summarize_recent(history, n=10):
    recent = history[-n:]
    games = len(recent)

    if games == 0:
        return {
            "games_available": 0,
            "win_rate": np.nan,
            "draw_rate": np.nan,
            "loss_rate": np.nan,
            "avg_goals_for": np.nan,
            "avg_goals_against": np.nan,
            "avg_goal_diff": np.nan,
        }

    goals_for = np.array([match["gf"] for match in recent], dtype=float)
    goals_against = np.array([match["ga"] for match in recent], dtype=float)
    outcomes = [match["outcome"] for match in recent]

    return {
        "games_available": games,
        "win_rate": outcomes.count("win") / games,
        "draw_rate": outcomes.count("draw") / games,
        "loss_rate": outcomes.count("loss") / games,
        "avg_goals_for": goals_for.mean(),
        "avg_goals_against": goals_against.mean(),
        "avg_goal_diff": (goals_for - goals_against).mean(),
    }


In [5]:
team_histories = defaultdict(list)
feature_rows = []

for _, row in results.iterrows():
    home_team = row["home_team"]
    away_team = row["away_team"]

    home_stats = summarize_recent(team_histories[home_team], n=10)
    away_stats = summarize_recent(team_histories[away_team], n=10)

    new_row = row.to_dict()

    for stat_name, stat_value in home_stats.items():
        new_row[f"home_{stat_name}"] = stat_value

    for stat_name, stat_value in away_stats.items():
        new_row[f"away_{stat_name}"] = stat_value

    feature_rows.append(new_row)

    if row["home_score"] > row["away_score"]:
        home_outcome = "win"
        away_outcome = "loss"
    elif row["home_score"] < row["away_score"]:
        home_outcome = "loss"
        away_outcome = "win"
    else:
        home_outcome = "draw"
        away_outcome = "draw"

    team_histories[home_team].append({
        "date": row["date"],
        "gf": row["home_score"],
        "ga": row["away_score"],
        "outcome": home_outcome,
    })

    team_histories[away_team].append({
        "date": row["date"],
        "gf": row["away_score"],
        "ga": row["home_score"],
        "outcome": away_outcome,
    })

model_data = pd.DataFrame(feature_rows)

model_data.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,total_goals,...,home_avg_goals_for,home_avg_goals_against,home_avg_goal_diff,away_games_available,away_win_rate,away_draw_rate,away_loss_rate,away_avg_goals_for,away_avg_goals_against,away_avg_goal_diff
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,0,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,6,...,0.000000,0.000000,0.000000,1,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,3,...,1.000000,2.000000,-1.000000,2,0.500000,0.500000,0.000000,2.000000,1.000000,1.000000
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,4,...,1.666667,1.333333,0.333333,3,0.333333,0.333333,0.333333,1.333333,1.666667,-0.333333
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,3,...,1.500000,1.750000,-0.250000,4,0.250000,0.500000,0.250000,1.750000,1.500000,0.250000


In [7]:
rows_before = len(model_data)

model_data = model_data[
    (model_data["home_games_available"] >= 5) &
    (model_data["away_games_available"] >= 5)].copy()

rows_after = len(model_data)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)


Rows before: 49405
Rows after: 48091
Rows removed: 1314


In [8]:
model_data["win_rate_diff"] = (
    model_data["home_win_rate"] - model_data["away_win_rate"])

model_data["draw_rate_diff"] = (
    model_data["home_draw_rate"] - model_data["away_draw_rate"])

model_data["loss_rate_diff"] = (
    model_data["home_loss_rate"] - model_data["away_loss_rate"])

model_data["goals_for_diff"] = (
    model_data["home_avg_goals_for"] - model_data["away_avg_goals_for"])

model_data["goals_against_diff"] = (
    model_data["home_avg_goals_against"] -
    model_data["away_avg_goals_against"])

model_data["goal_diff_recent_diff"] = (
    model_data["home_avg_goal_diff"] - model_data["away_avg_goal_diff"])


In [9]:
model_data["neutral"] = model_data["neutral"].astype(int)

model_data["home_field_advantage"] = (
    (model_data["neutral"] == 0) &
    (model_data["country"] == model_data["home_team"])
).astype(int)

model_data["is_world_cup"] = (
    model_data["tournament"] == "FIFA World Cup"
).astype(int)


In [10]:
MODEL_FEATURES = [
    "home_win_rate",
    "away_win_rate",
    "home_draw_rate",
    "away_draw_rate",
    "home_loss_rate",
    "away_loss_rate",
    "home_avg_goals_for",
    "away_avg_goals_for",
    "home_avg_goals_against",
    "away_avg_goals_against",
    "home_avg_goal_diff",
    "away_avg_goal_diff",
    "win_rate_diff",
    "draw_rate_diff",
    "loss_rate_diff",
    "goals_for_diff",
    "goals_against_diff",
    "goal_diff_recent_diff",
    "neutral",
    "home_field_advantage",
    "is_world_cup",
]


In [11]:
model_data[MODEL_FEATURES].isna().sum()


home_win_rate             0
away_win_rate             0
home_draw_rate            0
away_draw_rate            0
home_loss_rate            0
away_loss_rate            0
home_avg_goals_for        0
away_avg_goals_for        0
home_avg_goals_against    0
away_avg_goals_against    0
home_avg_goal_diff        0
away_avg_goal_diff        0
win_rate_diff             0
draw_rate_diff            0
loss_rate_diff            0
goals_for_diff            0
goals_against_diff        0
goal_diff_recent_diff     0
neutral                   0
home_field_advantage      0
is_world_cup              0
dtype: int64

In [13]:
model_data_path = PROCESSED_DIR / "model_data.csv"
features_path = PROCESSED_DIR / "model_features.json"

model_data.to_csv(model_data_path, index=False)

with open(features_path, "w") as f:
    json.dump(MODEL_FEATURES, f, indent=2)



In [14]:
latest_team_rows = []

for team, history in team_histories.items():
    stats = summarize_recent(history, n=10)
    latest_team_rows.append({"team": team, **stats})

latest_team_stats = pd.DataFrame(latest_team_rows)
latest_team_stats = latest_team_stats.sort_values("team").reset_index(drop=True)

latest_team_stats_path = PROCESSED_DIR / "latest_team_stats_pre_wc.csv"
latest_team_stats.to_csv(latest_team_stats_path, index=False)

latest_team_stats.head()


,team,games_available,win_rate,draw_rate,loss_rate,avg_goals_for,avg_goals_against,avg_goal_diff
0,Abkhazia,10,0.5,0.4,0.1,1.8,0.7,1.1
1,Afghanistan,10,0.1,0.2,0.7,0.4,1.5,-1.1
2,Albania,10,0.5,0.0,0.5,0.9,0.9,0.0
3,Alderney,10,0.2,0.0,0.8,0.9,3.2,-2.3
4,Algeria,10,0.7,0.2,0.1,2.1,0.4,1.7


### Feature Engineering Summary

This notebook creates rolling team-form features using only matches played before each row. The main features include recent win rate, draw rate, loss rate, goals scored, goals allowed, and goal difference for both teams.

The notebook also created matchup difference features and context indicators for neutral venues, home-field advantages, and World Cup matches.

Final outputs:

- /data/processed/model_data.csv
- /data/processed/model_features.json
- /data/processed/latest_team_stats_pre_wc.csv